In [6]:
!pip install keras_tuner

In [7]:
import pandas as pd
from tensorflow import keras
from tensorflow.keras import layers
from kerastuner.tuners import RandomSearch

In [8]:
df=pd.read_csv('Real_Combine.csv')

In [10]:
df.head()

,T,TM,Tm,SLP,H,VV,V,VM,PM 2.5
0,7.4,9.8,4.8,1017.6,93.0,0.5,4.3,9.4,219.720833
1,7.8,12.7,4.4,1018.5,87.0,0.6,4.4,11.1,182.187500
2,6.7,13.4,2.4,1019.4,82.0,0.6,4.8,11.1,154.037500
3,8.6,15.5,3.3,1018.7,72.0,0.8,8.1,20.6,223.208333
4,12.4,20.9,4.4,1017.3,61.0,1.3,8.7,22.2,200.645833


In [11]:
X=df.iloc[:,:-1] ## independent features
y=df.iloc[:,-1] ## dependent features

In [12]:

# Hyperparameters

#     How many number of hidden layers we should have?
#     How many number of neurons we should have in hidden layers?
#     What should be the Learning Rate


def build_model(hp):
    model = keras.Sequential()
    for i in range(hp.Int('num_layers', 2, 20)):
        model.add(layers.Dense(units=hp.Int('units_' + str(i),
                                            min_value=32,
                                            max_value=512,
                                            step=32),
                               activation='relu'))
    model.add(layers.Dense(1, activation='linear'))
    model.compile(
        optimizer=keras.optimizers.Adam(
            hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])),
        loss='mean_absolute_error',
        metrics=['mean_absolute_error'])
    return model

In [13]:
tuner = RandomSearch(
    build_model,
    objective='val_mean_absolute_error',
    max_trials=5,
    executions_per_trial=3,
    directory='project',
    project_name='Air Quality Index')

In [14]:
tuner.search_space_summary()

Search space summary
Default search space size: 4
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 2, 'max_value': 20, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


In [15]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)


In [16]:
tuner.search(X_train, y_train,
             epochs=5,
             validation_data=(X_test, y_test))

Trial 2 Complete [00h 00m 34s]
val_mean_absolute_error: nan

Best val_mean_absolute_error So Far: nan
Total elapsed time: 00h 01m 10s

Search: Running Trial #3

Value             |Best Value So Far |Hyperparameter
10                |18                |num_layers
320               |384               |units_0
480               |352               |units_1
0.001             |0.001             |learning_rate
512               |32                |units_2
96                |32                |units_3
160               |32                |units_4
416               |32                |units_5
288               |32                |units_6
96                |32                |units_7
352               |32                |units_8
480               |32                |units_9
224               |32                |units_10
64                |32                |units_11
512               |32                |units_12
64                |32                |units_13
320               |32                

/usr/local/lib/python3.11/dist-packages/keras_tuner/src/engine/metrics_tracking.py:111: RuntimeWarning: All-NaN axis encountered
  np.nanmin(values) if self.direction == "min" else np.nanmax(values)


RuntimeError: Number of consecutive failures exceeded the limit of 3.


In [18]:
tuner.results_summary()

Results summary
Results in project/Air Quality Index
Showing 10 best trials
Objective(name="val_mean_absolute_error", direction="min")

Trial 0 summary
Hyperparameters:
num_layers: 18
units_0: 384
units_1: 352
learning_rate: 0.001
units_2: 32
units_3: 32
units_4: 32
units_5: 32
units_6: 32
units_7: 32
units_8: 32
units_9: 32
units_10: 32
units_11: 32
units_12: 32
units_13: 32
units_14: 32
units_15: 32
units_16: 32
units_17: 32
Score: nan

Trial 1 summary
Hyperparameters:
num_layers: 13
units_0: 416
units_1: 512
learning_rate: 0.0001
units_2: 32
units_3: 384
units_4: 352
units_5: 192
units_6: 96
units_7: 416
units_8: 64
units_9: 448
units_10: 352
units_11: 96
units_12: 160
units_13: 32
units_14: 32
units_15: 160
units_16: 288
units_17: 448
Score: nan

Trial 2 summary
Hyperparameters:
num_layers: 10
units_0: 320
units_1: 480
learning_rate: 0.001
units_2: 512
units_3: 96
units_4: 160
units_5: 416
units_6: 288
units_7: 96
units_8: 352
units_9: 480
units_10: 224
units_11: 64
units_12: 512
u

In [17]:
import numpy as np

# Check for infinite values
print("Infinite values in X_train:", np.isinf(X_train).sum().sum())
print("Infinite values in X_test:", np.isinf(X_test).sum().sum())
print("Infinite values in y_train:", np.isinf(y_train).sum())
print("Infinite values in y_test:", np.isinf(y_test).sum())

# Check for missing values
print("Missing values in X_train:", X_train.isnull().sum().sum())
print("Missing values in X_test:", X_test.isnull().sum().sum())
print("Missing values in y_train:", y_train.isnull().sum())
print("Missing values in y_test:", y_test.isnull().sum())

Infinite values in X_train: 0
Infinite values in X_test: 0
Infinite values in y_train: 0
Infinite values in y_test: 0
Missing values in X_train: 0
Missing values in X_test: 0
Missing values in y_train: 1
Missing values in y_test: 0
